# Notebook to clean the movies data and load the database

In [1]:
import numpy as np
import pandas as pd
import ast

## Read movies.csv in a DataFrame

In [2]:
path_csv = "data/raw/movies.csv"
df_movies = pd.read_csv(path_csv)
display(df_movies.head())

,id,title,release_date,vote_average,vote_count
0,9479,The Nightmare Before Christmas,1993-10-09,7.852,10786
1,299536,Avengers: Infinity War,2018-04-25,8.246,33228
2,299534,Avengers: Endgame,2019-04-24,8.247,29087
3,157336,Interstellar,2014-11-04,8.500,41598
4,354912,Coco,2017-10-27,8.201,21602


## Read extra_info.csv in a DataFrame

In [3]:
path_csv = "data/raw/extra_info.csv"
df_extra_info = pd.read_csv(path_csv)
display(df_extra_info.head())

,id,budget,revenue,origin_country,runtime,genres,directors,cast
0,9479,18000000,75634409,US,76,"['Fantasy', 'Animation', 'Family']","[[57646, 'Henry Selick', 'Henry Selick', 2]]","[[531, 2, 'Danny Elfman', 'Danny Elfman', 'Jac..."
1,299536,300000000,2052415039,US,149,"['Adventure', 'Action', 'Science Fiction']","[[19271, 'Anthony Russo', 'Anthony Russo', 2],...","[[3223, 2, 'Robert Downey Jr.', 'Robert Downey..."
2,299534,356000000,2938567784,US,181,"['Adventure', 'Science Fiction', 'Action']","[[19271, 'Anthony Russo', 'Anthony Russo', 2],...","[[3223, 2, 'Robert Downey Jr.', 'Robert Downey..."
3,157336,165000000,746606706,US,169,"['Adventure', 'Drama', 'Science Fiction']","[[525, 'Christopher Nolan', 'Christopher Nolan...","[[10297, 2, 'Matthew McConaughey', 'Matthew Mc..."
4,354912,175000000,814641172,US,105,"['Family', 'Animation', 'Music', 'Adventure']","[[8, 'Lee Unkrich', 'Lee Unkrich', 2]]","[[1193391, 2, 'Anthony Gonzalez', 'Anthony Gon..."


## Join Dataframes

In [4]:
df_final = df_movies.merge(df_extra_info, on="id", how="left")

## Clean data

In [5]:
# Replace 0 with None in numeric columns
df_final["budget"] = df_final["budget"].replace(0, None)
df_final["revenue"] = df_final["revenue"].replace(0, None)
df_final["runtime"] = df_final["runtime"].replace(0, None)

# Change 'release_date' type to datetime
df_final["release_date"] = pd.to_datetime(df_final["release_date"])

In [6]:
# Set directors, cast and genres as list
def parse_list(value):
    ''' Turn the string "['A', 'B']" stored in the CSV back into a list (None if it is not valid). '''
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return None

for column in ["directors", "cast", "genres"]:
    df_final[column] = df_final[column].apply(parse_list)

In [7]:
print(df_final.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              200 non-null    int64         
 1   title           200 non-null    object        
 2   release_date    200 non-null    datetime64[ns]
 3   vote_average    200 non-null    float64       
 4   vote_count      200 non-null    int64         
 5   budget          200 non-null    int64         
 6   revenue         200 non-null    int64         
 7   origin_country  200 non-null    object        
 8   runtime         200 non-null    int64         
 9   genres          200 non-null    object        
 10  directors       200 non-null    object        
 11  cast            200 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(5), object(5)
memory usage: 18.9+ KB
None


In [8]:
display(df_final.head())

,id,title,release_date,vote_average,vote_count,budget,revenue,origin_country,runtime,genres,directors,cast
0,9479,The Nightmare Before Christmas,1993-10-09,7.852,10786,18000000,75634409,US,76,"[Fantasy, Animation, Family]","[[57646, Henry Selick, Henry Selick, 2]]","[[531, 2, Danny Elfman, Danny Elfman, Jack Ske..."
1,299536,Avengers: Infinity War,2018-04-25,8.246,33228,300000000,2052415039,US,149,"[Adventure, Action, Science Fiction]","[[19271, Anthony Russo, Anthony Russo, 2], [19...","[[3223, 2, Robert Downey Jr., Robert Downey Jr..."
2,299534,Avengers: Endgame,2019-04-24,8.247,29087,356000000,2938567784,US,181,"[Adventure, Science Fiction, Action]","[[19271, Anthony Russo, Anthony Russo, 2], [19...","[[3223, 2, Robert Downey Jr., Robert Downey Jr..."
3,157336,Interstellar,2014-11-04,8.500,41598,165000000,746606706,US,169,"[Adventure, Drama, Science Fiction]","[[525, Christopher Nolan, Christopher Nolan, 2]]","[[10297, 2, Matthew McConaughey, Matthew McCon..."
4,354912,Coco,2017-10-27,8.201,21602,175000000,814641172,US,105,"[Family, Animation, Music, Adventure]","[[8, Lee Unkrich, Lee Unkrich, 2]]","[[1193391, 2, Anthony Gonzalez, Anthony Gonzal..."


In [48]:
def clean_df_people(df):
    ''' Function that recive a dataframe, delete duplicate rows base on 'id', and set to null 'original_name' if equal to 'name'. '''
    
    df.drop_duplicates(subset="id", keep='first', inplace=True)
    
    df["original_name"] = df["original_name"].mask(df["original_name"].eq(df["name"]))

    return df

## Get all diferent directors

In [49]:
df_directors = pd.DataFrame(df_final["directors"].explode().tolist(), columns=["id", "name", "original_name", "gender"])
df_directors = clean_df_people(df_directors)

In [50]:
display(df_directors.head())

,id,name,original_name,gender
0,57646,Henry Selick,NaN,0
1,19271,Anthony Russo,NaN,0
2,19272,Joe Russo,NaN,0
5,525,Christopher Nolan,NaN,0
6,8,Lee Unkrich,NaN,0


In [38]:
## Save df in a CSV
path_csv = "data/directors.csv"
df_directors.to_csv(path_csv, sep = ",", header = True, index = False, mode = "a", na_rep = "N/A")

## Get all diferent actors

In [39]:
# DF with all the info
df_cast = pd.DataFrame(df_final["cast"].explode().tolist(), columns=["id", "gender", "name", "original_name", "character"])

In [51]:
# DF with only the diferent actors info
df_actors = df_cast.drop(["character"], axis=1, inplace=False)
df_actors = clean_df_people(df_actors)

In [52]:
display(df_actors.head())

,id,gender,name,original_name
0,531,0,Danny Elfman,NaN
1,14541,0,Chris Sarandon,NaN
2,11514,1,Catherine O'Hara,NaN
3,21282,0,William Hickey,NaN
4,13243,0,Glenn Shadix,NaN


In [53]:
## Save df in a CSV
path_csv = "data/actors.csv"
df_actors.to_csv(path_csv, sep = ",", header = True, index = False, mode = "a", na_rep = "N/A")

## Set Database info

In [9]:
import os
import pyodbc

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

In [10]:
SERVER = os.getenv("SERVER")
USER = os.getenv("USER")
PASSWORD = os.getenv("PASSWORD")
DATABASE = os.getenv("DATABASE")
PORT = os.getenv("PORT")

In [ ]:
# Create conection
DRIVER = '{ODBC Driver 17 for SQL Server}'
conn = pyodbc.connect(f'DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};UID={USER};PWD={PASSWORD}')

# Create cursor
cursor = conn.cursor()

### Load genres

### Load directors

### Load actors

### Load movies, characters and n:m tables

In [ ]:
conn.close()